# Word2Vec Tasks
## Completed by William Brannock (svv8fs)

This notebook contains the word2vec tasks for the Final Project. This notebook was guided by this class notebook. https://www.kaggle.com/code/ontoligent/uva-ds-5001-m09-word2vec-with-wine-reviews

# Set Up

In [15]:
import contextlib
import io
import warnings

import gensim
import pandas as pd
import plotly.express as px
import plotly.io as pio
from gensim.corpora import Dictionary
from gensim.models import word2vec
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore", category=FutureWarning)
pio.renderers.default = "plotly_mimetype+notebook_connected"

In [16]:
gensim.__version__

'4.4.0'

# Config

In [17]:
data_home = "."
output_dir = "output"
data_prefix = "constitutions"
source_dir = output_dir
CSV_DELIM = "|"

OHCO = ["constitution_id", "provision_num", "para_num", "sent_num", "token_num"]
BAG = OHCO[:4]
bag_name = "SENTS"

vector_size = 200
min_count = 50
random_state = 23

# Import Tables

In [18]:
# Read in corpus
CORPUS = pd.read_csv(
    f"{source_dir}/{data_prefix}-CORPUS.csv",
    sep=CSV_DELIM,
    keep_default_na=False,
    usecols=OHCO + ["term_str"],
)

CORPUS.shape

(4296300, 6)

# Prepare Sentence Bags


In [19]:
TOKENS = CORPUS[CORPUS.term_str != ""].copy()

docs_by_bag = TOKENS.groupby(BAG, sort=False).term_str.apply(list)
docs = [terms for terms in docs_by_bag.tolist() if len(terms) > 1]



In [20]:
docs[:5]

[['in',
  'the',
  'name',
  'of',
  'allah',
  'the',
  'most',
  'beneficent',
  'the',
  'most',
  'merciful'],
 ['praise',
  'be',
  'to',
  'allah',
  'the',
  'cherisher',
  'and',
  'sustainer',
  'of',
  'worlds',
  'and',
  'praise',
  'and',
  'peace',
  'be',
  'upon',
  'mohammad',
  'his',
  'last',
  'messenger',
  'and',
  'his',
  'disciples',
  'and',
  'followers'],
 ['we', 'the', 'people', 'of', 'afghanistan'],
 ['believing',
  'firmly',
  'in',
  'almighty',
  'god',
  'relying',
  'on',
  'his',
  'divine',
  'will',
  'and',
  'adhering',
  'to',
  'the',
  'holy',
  'religion',
  'of',
  'islam'],
 ['realizing',
  'the',
  'previous',
  'injustices',
  'miseries',
  'and',
  'innumerable',
  'disasters',
  'which',
  'have',
  'befallen',
  'our',
  'country']]

# Generate Word Embeddings

In [21]:
dictionary = Dictionary(docs)

w2v_params = dict(
    vector_size=vector_size,
    window=2,
    min_count=min_count,
    workers=4,
)

stderr_buffer = io.StringIO()
with contextlib.redirect_stderr(stderr_buffer):
    model = word2vec.Word2Vec(docs, **w2v_params)

model

# VOCAB_W2V

In [22]:
vector_cols = [f"w2v_{i:03d}" for i in range(model.wv.vector_size)]

WV = pd.DataFrame(model.wv.vectors, index=model.wv.index_to_key, columns=vector_cols)
WV.index.name = "term_str"

VOCAB_W2V = WV.copy()
VOCAB_W2V.head()

,w2v_000,w2v_001,w2v_002,w2v_003,w2v_004,w2v_005,w2v_006,w2v_007,w2v_008,w2v_009,...,w2v_190,w2v_191,w2v_192,w2v_193,w2v_194,w2v_195,w2v_196,w2v_197,w2v_198,w2v_199
term_str,,,,,,,,,,,,,,,,,,,,,
the,0.148620,0.020840,-0.558895,0.003606,-0.756536,0.424363,0.487369,0.073251,0.123551,-0.205128,...,-0.062787,0.091575,-0.482527,-0.256216,-0.260501,-0.562191,-0.165395,0.170800,-0.261442,0.469291
of,0.114963,-0.412362,0.009254,0.327893,0.157621,0.142745,0.033958,0.375589,-0.306988,-0.725418,...,-0.178867,0.094422,0.309221,-0.483708,0.742383,-0.271363,-0.195627,-0.638369,-1.563962,0.601229
and,-0.330278,0.381994,-0.711232,0.680507,0.575108,0.318160,0.523621,0.133328,-0.018284,0.185316,...,-0.321035,-0.189255,0.554417,-0.060297,0.129754,1.156704,-0.888393,0.292796,0.152109,0.353142
to,0.768105,0.704476,0.590598,0.471852,0.296063,1.171606,1.376678,-1.508775,-0.867998,-0.124574,...,-1.081811,-0.446840,-0.518363,0.837286,0.656134,-0.003107,-0.011213,0.123462,1.164938,-0.546595
in,-0.839762,0.025101,0.811526,0.425650,0.021260,0.836877,-0.024591,-0.212845,-0.241834,-0.142137,...,-0.992648,1.207361,1.190179,0.956382,0.941836,0.163697,-1.004480,1.210818,-0.690301,0.399328


# t-SNE Plot

In [23]:
tsne_vectors = VOCAB_W2V[vector_cols]
perplexity = 40

tsne_engine = TSNE(
    perplexity=perplexity,
    n_components=2,
    init="pca",
    max_iter=2500,
    random_state=random_state,
    learning_rate="auto",
)

TSNE_W2V = pd.DataFrame(
    tsne_engine.fit_transform(tsne_vectors),
    columns=["x", "y"],
    index=tsne_vectors.index,
)
TSNE_W2V.index.name = "term_str"
TSNE_W2V.head()

,x,y
term_str,,
the,8.103005,5.087724
of,-9.476609,3.143704
and,-3.792300,-42.204483
to,29.512796,2.151423
in,33.185394,5.442095


In [24]:
fig = px.scatter(
    TSNE_W2V.reset_index(),
    x="x",
    y="y",
    hover_name="term_str",
    text="term_str",
    height=900,
    width=1100,
    title="Word2Vec Term Embeddings Projected with t-SNE",
    template="plotly_white",
)
fig.update_traces(marker={"opacity": 0.65, "line": {"width": 0.4, "color": "#333333"}})
fig.update_traces(textposition="top center")
fig.update_layout(xaxis_title="t-SNE 1", yaxis_title="t-SNE 2")
fig.show()

# Save

In [25]:
save_path = f"{output_dir}/{data_prefix}"

VOCAB_W2V.to_csv(f"{save_path}-VOCAB_W2V-{bag_name}.csv", sep=CSV_DELIM)